# 01 — Comparación Nacional vs Regional

Compara el escenario **SAND Nacional** contra el **Regional** (7 regiones con prefijo
en `TECHNOLOGY`/`FUEL`), conducido por los índices de cada parámetro según
`config_depurado.yaml`:

- **Aditivos**: nacional == suma de las regiones (excluyendo centinelas 99999/solo-9s).
- **Intensivos** (`parametros_intensivos` en `config/params_config.yaml`): cada región
  debe conservar el valor nacional.
- Parámetros sin `YEAR` se comparan por `Time indipendent variables`.

Genera un Excel con hojas **Comparacion** (resaltado rojo sobre el umbral),
**Anomalias** y **Resumen**, y una gráfica Top-N exportada a PNG.

> Ejecutable de principio a fin con *Run All*.

In [ ]:
# --- Setup: raíz del proyecto, módulos src/ y configuración ---
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda v: f"{v:,.6f}")

RAIZ = Path.cwd().resolve()
if not (RAIZ / "config").exists():   # ejecutado desde notebooks/
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ / "src"))

import comparador
import regionalizador
import reporte
import sand_io
import utils
import yaml_parser

utils.configurar_logging()

params_cfg = yaml_parser.cargar_params_config(RAIZ / "config" / "params_config.yaml")
paths = yaml_parser.cargar_paths_config(RAIZ / "config" / "paths_config.yaml", raiz=RAIZ)
otoole = yaml_parser.cargar_config_otoole(paths["escenario_nacional"]["config_yaml"])

print(f"Nacional : {paths['escenario_nacional']['sand'].name}")
print(f"Regional : {paths['escenario_regional']['sand'].name}")
print(f"Config otoole: {len(otoole['param'])} parámetros, {len(otoole['set'])} sets")
print(f"Intensivos: {len(params_cfg['parametros_intensivos'])} | Centinela: {params_cfg['valor_centinela']}")

## 1. Carga de los archivos SAND

In [ ]:
df_nacional = sand_io.cargar_sand(paths["escenario_nacional"]["sand"])
df_regional = sand_io.cargar_sand(paths["escenario_regional"]["sand"])

anios = sand_io.columnas_anio(df_nacional)
display(pd.DataFrame([
    {"Archivo": "Nacional", "Filas": len(df_nacional),
     "Parametros": df_nacional["Parameter"].nunique(),
     "TECHNOLOGY": df_nacional["TECHNOLOGY"].nunique(), "FUEL": df_nacional["FUEL"].nunique()},
    {"Archivo": "Regional", "Filas": len(df_regional),
     "Parametros": df_regional["Parameter"].nunique(),
     "TECHNOLOGY": df_regional["TECHNOLOGY"].nunique(), "FUEL": df_regional["FUEL"].nunique()},
]))
print(f"Años: {anios[0]}–{anios[-1]} ({len(anios)})")

## 2. Filtros de la comparación

- `MODO_COMPARACION`: `'general'` (todos los parámetros), `'parametro'` o `'lista_parametros'`.
- `TECNOLOGIAS_A_FILTRAR` / `FUELS_A_FILTRAR` solo actúan sobre parámetros indexados
  por esa dimensión; `MODO_FILTRO` = `'exacto'` o `'contiene'`.

In [ ]:
MODO_COMPARACION = "lista_parametros"   # 'general' | 'parametro' | 'lista_parametros'
PARAMETROS_FILTRO = ["AccumulatedAnnualDemand", "TotalTechnologyAnnualActivityLowerLimit"]
TECNOLOGIAS_A_FILTRAR = []              # solo activo si el parámetro se indexa por TECHNOLOGY
FUELS_A_FILTRAR = []                    # solo activo si el parámetro se indexa por FUEL
MODO_FILTRO = "exacto"                  # 'exacto' | 'contiene'

TOP_N = params_cfg["top_n_grafica"]     # Top-N de la gráfica/tabla de diferencias
UMBRAL_ALERTA = params_cfg["umbral_alerta_pct"]

## 3. Comparación

`Región = SUMA_REGIONES` en aditivos; nombre del prefijo regional en intensivos.
El resultado viene ordenado por `|Diferencia|` descendente.

In [ ]:
resultado = comparador.comparar_escenarios(
    df_nacional, df_regional, otoole["param"], params_cfg,
    modo=MODO_COMPARACION,
    parametros_filtro=PARAMETROS_FILTRO,
    tecnologias=TECNOLOGIAS_A_FILTRAR,
    fuels=FUELS_A_FILTRAR,
    modo_filtro=MODO_FILTRO,
)
comparacion = resultado["comparacion"]
discrepancias = resultado["discrepancias"]
anomalias = resultado["anomalias"]

print(reporte.resumen_discrepancias(discrepancias, len(comparacion),
                                    params_cfg["tolerancia_comparacion"]))
display(Markdown("**Resumen por parámetro**"))
display(resultado["resumen"])

In [ ]:
display(Markdown(f"**Anomalías detectadas: {len(anomalias)}**"))
if not anomalias.empty:
    display(anomalias.groupby("Tipo_Anomalia", as_index=False).size())
    display(anomalias.head(30))

In [ ]:
display(Markdown("**Mayores discrepancias**"))
display(discrepancias.head(30))

## 4. Gráfica y tabla Top-N (solo en modo `parametro` / `lista_parametros`)

In [ ]:
RUTA_PNG = paths["outputs"]["reportes"] / "top_diferencias.png"

if MODO_COMPARACION != "general":
    display(Markdown(f"**Top-{TOP_N} diferencias absolutas por tecnología/fuel**"))
    display(reporte.top_diferencias(comparacion, n=TOP_N))
    reporte.grafica_top_diferencias(comparacion, n=TOP_N, ruta_png=RUTA_PNG)
else:
    reporte.grafica_discrepancias_por_parametro(discrepancias)

## 5. Exportar reporte

Hojas: **Comparacion** (filas con `|Diferencia %| >` umbral resaltadas en rojo),
**Anomalias** y **Resumen**.

In [ ]:
ruta_reporte = reporte.exportar_reporte_comparacion(
    paths["outputs"]["reportes"] / "Reporte_Comparacion_Nacional_vs_Regional.xlsx",
    comparacion, anomalias, resultado["resumen"],
    umbral_alerta_pct=UMBRAL_ALERTA,
)
print(f"Reporte : {ruta_reporte}")
if MODO_COMPARACION != "general":
    print(f"Gráfica : {RUTA_PNG}")

---
*Smoke tests de los módulos: `python tests/test_smoke.py` desde la raíz del proyecto.*